# Lab E — Contract Intelligence Business App

## Business-consumption architecture

```text
                          BUSINESS USER
                               │
                               ▼
                    ┌────────────────────┐
                    │  DATABRICKS APP    │
                    │    Streamlit       │
                    └─────────┬──────────┘
                              │
          ┌───────────────────┼────────────────────┐
          ▼                   ▼                    ▼
   Executive Overview     Contract 360      Ask Contract Intelligence
          │                   │                    │
          ▼                   ▼                    ▼
      GOLD VIEW           GOLD + SILVER      Gold facts + Silver evidence
                                                   │
                                                   ▼
                                          Follow-up proposal
                                                   │
                                                   ▼
                                            Approval Inbox
                                                   │
                                         ┌─────────┴─────────┐
                                         ▼                   ▼
                                      REJECT              APPROVE
                                                              │
                                                              ▼
                                                         Action Queue

                    AI/BI Dashboard ── linked from app

                    Genie One ── enterprise business entry point
```

## Objective

Create the **business-facing consumption layer** for the Enterprise Contract Intelligence POC.

Lab E does not rebuild the Medallion or agent foundation.

It consumes the outputs from:

- **Lab C** — Bronze → Silver → Gold contract intelligence
- **Lab D** — evaluation gate, approval model, and action queue

The notebook generates a deployable Streamlit Databricks App with:

1. **Executive Overview**
2. **Contract 360**
3. **Ask Contract Intelligence**
4. **Approval Inbox**
5. **Action Queue**
6. **AI/BI Dashboard link**

## Security model

The generated app does **not** contain:
- PAT tokens
- passwords
- client secrets
- bearer tokens

It uses Databricks App authorization and a SQL warehouse resource.

The SQL warehouse ID is supplied to the app through the resource key `sql-warehouse`, rather than being hard-coded.


## Section 1 — Validate Lab C and Lab D Assets

In [0]:
# Validate that the Medallion and governance assets required by the business application are available.
row = spark.sql(
    "SELECT current_catalog() AS catalog_name, current_schema() AS schema_name"
).first()

CATALOG = row["catalog_name"]
SCHEMA = row["schema_name"]

GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_enterprise_contracts"
SILVER_CLAUSES = f"{CATALOG}.{SCHEMA}.silver_contract_clauses"
APPROVAL_TABLE = f"{CATALOG}.{SCHEMA}.contract_agent_approvals"
ACTION_QUEUE_TABLE = f"{CATALOG}.{SCHEMA}.contract_agent_action_queue"

required_data_objects = [
    GOLD_TABLE,
    SILVER_CLAUSES,
]

missing_data_objects = [
    table_name
    for table_name in required_data_objects
    if not spark.catalog.tableExists(table_name)
]

if missing_data_objects:
    raise RuntimeError(
        "Run Lab C first. Missing objects: "
        + ", ".join(missing_data_objects)
    )

print("Lab C Medallion assets detected.")
print("Gold:", GOLD_TABLE)
print("Silver evidence:", SILVER_CLAUSES)


In [0]:
# Ensure the Lab D approval and action-queue tables exist with the expected governance schema.
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {APPROVAL_TABLE} (
        approval_id STRING,
        action_id STRING,
        contract_id STRING,
        supplier_name STRING,
        proposed_action STRING,
        action_owner STRING,
        priority STRING,
        rationale STRING,
        evidence STRING,
        requested_at TIMESTAMP,
        approval_status STRING,
        approver STRING,
        reviewed_at TIMESTAMP
    )
    USING DELTA
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {ACTION_QUEUE_TABLE} (
        action_id STRING,
        approval_id STRING,
        contract_id STRING,
        supplier_name STRING,
        action_type STRING,
        action_owner STRING,
        proposed_action STRING,
        priority STRING,
        rationale STRING,
        evidence STRING,
        queue_status STRING,
        queued_at TIMESTAMP,
        completed_at TIMESTAMP
    )
    USING DELTA
    """
)

print("Governance tables ready.")
print("Approval table:", APPROVAL_TABLE)
print("Action queue:", ACTION_QUEUE_TABLE)


## Section 2 — Preview the Business Data

The application uses:

- **Gold** for business KPIs, Contract 360, milestones, payments, risks, and renewals.
- **Silver contract clauses** for contractual evidence.
- **Approval / Action Queue tables** for governed follow-up.


In [0]:
# Preview the Gold contract command-center data that the Streamlit application will expose.
from pyspark.sql import functions as F

display(
    spark.table(GOLD_TABLE)
    .select(
        "contract_id",
        "supplier_name",
        "contract_value",
        "currency",
        "risk_severity",
        "risk_type",
        "next_milestone",
        "next_milestone_date",
        "next_payment_amount",
        "next_payment_currency",
        "days_to_expiry",
        "renewal_review_due",
        "recommended_action",
    )
    .orderBy(
        F.col("risk_severity").desc(),
        F.col("next_milestone_date")
    )
)


## Section 3 — Business App Pages

The generated Streamlit app contains six business pages.

### Executive Overview
Portfolio KPIs, high-risk contracts, renewal exposure, supplier risk summary, and dashboard link.

### Contract 360
Select a supplier and contract to see:
- contract value
- risk
- SLA
- penalty
- next milestone
- next payment
- expiry
- obligations
- contract clauses

### Ask Contract Intelligence
Natural-language question combining:
- Gold facts
- Silver evidence
- `ai_query` reasoning where available

### Approval Inbox
Human reviewer sees:
- proposed action
- priority
- rationale
- evidence
- APPROVE / REJECT controls

### Action Queue
Approved follow-up actions.

### AI/BI Dashboard
Link to the published detailed portfolio dashboard.


## Section 4 — Generate the Databricks App Project

In [0]:
# Define the full Streamlit business application source that will be written into app.py.
app_py = 'import os\nimport re\nimport json\nimport uuid\nfrom datetime import datetime\n\nimport pandas as pd\nimport streamlit as st\nfrom databricks import sql\nfrom databricks.sdk.core import Config\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nst.set_page_config(\n    page_title="Enterprise Contract Intelligence",\n    page_icon="📄",\n    layout="wide",\n)\n\ncfg = Config()\n\nCATALOG = os.getenv("CATALOG_NAME", "workspace")\nSCHEMA = os.getenv("SCHEMA_NAME", "default")\nWAREHOUSE_ID = os.getenv("WAREHOUSE_ID")\nDASHBOARD_URL = os.getenv("DASHBOARD_URL", "").strip()\nLLM_ENDPOINT = os.getenv(\n    "LLM_ENDPOINT",\n    "system.ai.meta-llama-3-3-70b-instruct",\n)\n\nGOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_enterprise_contracts"\nSILVER_CLAUSES = f"{CATALOG}.{SCHEMA}.silver_contract_clauses"\nAPPROVAL_TABLE = f"{CATALOG}.{SCHEMA}.contract_agent_approvals"\nACTION_QUEUE_TABLE = f"{CATALOG}.{SCHEMA}.contract_agent_action_queue"\n\n\ndef sql_escape(value):\n    if value is None:\n        return ""\n    return str(value).replace("\'", "\'\'")\n\n\n@st.cache_resource\ndef get_connection():\n    if not WAREHOUSE_ID:\n        raise RuntimeError(\n            "WAREHOUSE_ID is not configured. Add a SQL warehouse "\n            "resource to the Databricks App using resource key sql-warehouse."\n        )\n\n    server_hostname = cfg.host\n    if server_hostname.startswith("https://"):\n        server_hostname = server_hostname.replace("https://", "")\n    elif server_hostname.startswith("http://"):\n        server_hostname = server_hostname.replace("http://", "")\n\n    return sql.connect(\n        server_hostname=server_hostname,\n        http_path=f"/sql/1.0/warehouses/{WAREHOUSE_ID}",\n        credentials_provider=lambda: cfg.authenticate,\n        _use_arrow_native_complex_types=False,\n    )\n\n\ndef query_df(query):\n    conn = get_connection()\n    with conn.cursor() as cursor:\n        cursor.execute(query)\n        return cursor.fetchall_arrow().to_pandas()\n\n\ndef execute_sql(statement):\n    conn = get_connection()\n    with conn.cursor() as cursor:\n        cursor.execute(statement)\n\n\n@st.cache_data(ttl=120)\ndef load_gold():\n    return query_df(\n        f"SELECT * FROM {GOLD_TABLE} ORDER BY contract_id"\n    )\n\n\n@st.cache_data(ttl=120)\ndef load_clauses():\n    return query_df(\n        f"""\n        SELECT\n            chunk_id,\n            contract_id,\n            section_number,\n            clause_type,\n            clause_text,\n            document_name\n        FROM {SILVER_CLAUSES}\n        ORDER BY contract_id, section_number\n        """\n    )\n\n\n@st.cache_resource\ndef build_retriever():\n    clauses = load_clauses()\n    vectorizer = TfidfVectorizer(\n        stop_words="english",\n        ngram_range=(1, 2),\n    )\n    matrix = vectorizer.fit_transform(\n        clauses["clause_text"].fillna("")\n    )\n    return clauses, vectorizer, matrix\n\n\ndef retrieve_evidence(query, contract_id=None, top_k=4):\n    clauses, vectorizer, matrix = build_retriever()\n    candidates = clauses.copy()\n\n    if contract_id:\n        candidates = candidates[\n            candidates["contract_id"] == contract_id\n        ].copy()\n\n    if candidates.empty:\n        return pd.DataFrame()\n\n    indices = candidates.index.to_list()\n    query_vector = vectorizer.transform([query])\n    scores = cosine_similarity(\n        query_vector,\n        matrix[indices],\n    ).flatten()\n\n    candidates["score"] = scores\n\n    return (\n        candidates.sort_values("score", ascending=False)\n        .head(top_k)\n        .reset_index(drop=True)\n    )\n\n\ndef call_ai_query(prompt):\n    prompt_sql = sql_escape(prompt)\n    endpoint_sql = sql_escape(LLM_ENDPOINT)\n\n    try:\n        result = query_df(\n            f"""\n            SELECT ai_query(\n                \'{endpoint_sql}\',\n                \'{prompt_sql}\'\n            ) AS response\n            """\n        )\n        return str(result.iloc[0]["response"])\n    except Exception:\n        return None\n\n\ndef identify_contract(question, gold):\n    match = re.search(\n        r"CTR-\\d{4}-\\d{3}",\n        question,\n        flags=re.I,\n    )\n\n    if match:\n        candidate = match.group(0).upper()\n        if candidate in set(gold["contract_id"].astype(str)):\n            return candidate\n\n    return None\n\n\ndef answer_contract_question(question):\n    gold = load_gold()\n    contract_id = identify_contract(question, gold)\n    question_lower = question.lower()\n\n    if contract_id:\n        facts = gold[\n            gold["contract_id"] == contract_id\n        ].copy()\n\n    elif "renew" in question_lower or "expir" in question_lower:\n        facts = gold[\n            (gold["days_to_expiry"] >= 0)\n            & (gold["days_to_expiry"] <= 120)\n        ].sort_values("days_to_expiry").head(10)\n\n    elif "payment" in question_lower:\n        facts = (\n            gold[\n                gold["next_payment_due_date"].notna()\n            ]\n            .sort_values("next_payment_due_date")\n            .head(10)\n        )\n\n    elif (\n        "risk" in question_lower\n        or "attention" in question_lower\n        or "intervention" in question_lower\n    ):\n        facts = (\n            gold[\n                gold["risk_severity"] == "HIGH"\n            ]\n            .sort_values("next_milestone_date")\n            .head(10)\n        )\n\n    else:\n        facts = (\n            gold.sort_values(\n                ["risk_severity", "next_milestone_date"],\n                ascending=[False, True],\n            )\n            .head(10)\n        )\n\n    focus_contract = (\n        contract_id\n        if contract_id\n        else (\n            str(facts.iloc[0]["contract_id"])\n            if not facts.empty\n            else None\n        )\n    )\n\n    evidence = retrieve_evidence(\n        question,\n        contract_id=focus_contract,\n        top_k=4,\n    )\n\n    facts_json = facts.to_dict(orient="records")\n    evidence_json = (\n        evidence[\n            [\n                "contract_id",\n                "clause_type",\n                "clause_text",\n                "score",\n            ]\n        ].to_dict(orient="records")\n        if not evidence.empty\n        else []\n    )\n\n    prompt = f"""\nYou are an enterprise Contract Intelligence assistant.\n\nUse only the supplied Gold contract facts and Silver contract evidence.\nDo not invent contract terms, dates, payment amounts, penalties, or actions.\nIf evidence is incomplete, say so.\n\nUSER QUESTION:\n{question}\n\nGOLD FACTS:\n{json.dumps(facts_json, default=str)}\n\nSILVER CONTRACT EVIDENCE:\n{json.dumps(evidence_json, default=str)}\n\nReturn a concise business response with:\n1. Contract / supplier\n2. Key finding\n3. Risk or exposure\n4. Recommended follow-up\n5. Supporting evidence\n"""\n\n    ai_response = call_ai_query(prompt)\n\n    if ai_response:\n        response = ai_response\n    elif not facts.empty:\n        row = facts.iloc[0]\n        response = (\n            f"Contract {row.get(\'contract_id\')} for "\n            f"{row.get(\'supplier_name\')} has risk "\n            f"{row.get(\'risk_type\')} / {row.get(\'risk_severity\')}. "\n            f"Next milestone: {row.get(\'next_milestone\')} "\n            f"({row.get(\'next_milestone_date\')}). "\n            f"Next payment: {row.get(\'next_payment_amount\')} "\n            f"{row.get(\'next_payment_currency\')}. "\n            f"Recommended follow-up: {row.get(\'recommended_action\')}."\n        )\n    else:\n        response = "No matching governed contract facts were found."\n\n    return {\n        "response": response,\n        "facts": facts,\n        "evidence": evidence,\n        "focus_contract_id": focus_contract,\n    }\n\n\ndef create_approval_request(\n    contract_id,\n    supplier_name,\n    proposed_action,\n    owner,\n    priority,\n    rationale,\n    evidence,\n):\n    approval_id = str(uuid.uuid4())\n    action_id = str(uuid.uuid4())\n\n    execute_sql(\n        f"""\n        INSERT INTO {APPROVAL_TABLE}\n        (\n            approval_id,\n            action_id,\n            contract_id,\n            supplier_name,\n            proposed_action,\n            action_owner,\n            priority,\n            rationale,\n            evidence,\n            requested_at,\n            approval_status,\n            approver,\n            reviewed_at\n        )\n        VALUES\n        (\n            \'{sql_escape(approval_id)}\',\n            \'{sql_escape(action_id)}\',\n            \'{sql_escape(contract_id)}\',\n            \'{sql_escape(supplier_name)}\',\n            \'{sql_escape(proposed_action)}\',\n            \'{sql_escape(owner)}\',\n            \'{sql_escape(priority)}\',\n            \'{sql_escape(rationale)}\',\n            \'{sql_escape(evidence)}\',\n            current_timestamp(),\n            \'PENDING\',\n            NULL,\n            NULL\n        )\n        """\n    )\n\n    return approval_id\n\n\ndef review_approval(\n    approval_id,\n    decision,\n    approver,\n):\n    decision = decision.upper()\n    status = "APPROVED" if decision == "APPROVE" else "REJECTED"\n\n    execute_sql(\n        f"""\n        UPDATE {APPROVAL_TABLE}\n        SET\n            approval_status = \'{status}\',\n            approver = \'{sql_escape(approver)}\',\n            reviewed_at = current_timestamp()\n        WHERE approval_id = \'{sql_escape(approval_id)}\'\n          AND approval_status = \'PENDING\'\n        """\n    )\n\n    if status != "APPROVED":\n        return\n\n    row = query_df(\n        f"""\n        SELECT *\n        FROM {APPROVAL_TABLE}\n        WHERE approval_id = \'{sql_escape(approval_id)}\'\n        LIMIT 1\n        """\n    )\n\n    if row.empty:\n        return\n\n    item = row.iloc[0]\n\n    existing = query_df(\n        f"""\n        SELECT COUNT(*) AS n\n        FROM {ACTION_QUEUE_TABLE}\n        WHERE action_id = \'{sql_escape(item["action_id"])}\'\n        """\n    )\n\n    if int(existing.iloc[0]["n"]) > 0:\n        return\n\n    execute_sql(\n        f"""\n        INSERT INTO {ACTION_QUEUE_TABLE}\n        (\n            action_id,\n            approval_id,\n            contract_id,\n            supplier_name,\n            action_type,\n            action_owner,\n            proposed_action,\n            priority,\n            rationale,\n            evidence,\n            queue_status,\n            queued_at,\n            completed_at\n        )\n        VALUES\n        (\n            \'{sql_escape(item["action_id"])}\',\n            \'{sql_escape(item["approval_id"])}\',\n            \'{sql_escape(item["contract_id"])}\',\n            \'{sql_escape(item["supplier_name"])}\',\n            \'CONTRACT_FOLLOW_UP\',\n            \'{sql_escape(item["action_owner"])}\',\n            \'{sql_escape(item["proposed_action"])}\',\n            \'{sql_escape(item["priority"])}\',\n            \'{sql_escape(item["rationale"])}\',\n            \'{sql_escape(item["evidence"])}\',\n            \'QUEUED\',\n            current_timestamp(),\n            NULL\n        )\n        """\n    )\n\n\ndef render_executive_overview():\n    st.header("Contract Command Center")\n\n    gold = load_gold()\n\n    total_contracts = len(gold)\n    total_value = float(gold["contract_value"].fillna(0).sum())\n    high_risk = int(\n        (gold["risk_severity"] == "HIGH").sum()\n    )\n    renewals_due = int(\n        gold["renewal_review_due"].fillna(False).astype(bool).sum()\n    )\n\n    c1, c2, c3, c4 = st.columns(4)\n    c1.metric("Contracts", f"{total_contracts:,}")\n    c2.metric("Portfolio Value", f"{total_value:,.0f}")\n    c3.metric("High-Risk Contracts", f"{high_risk:,}")\n    c4.metric("Renewal Reviews Due", f"{renewals_due:,}")\n\n    st.subheader("Contracts Requiring Attention")\n    attention = gold[\n        (gold["risk_severity"] == "HIGH")\n        | gold["renewal_review_due"].fillna(False).astype(bool)\n    ].copy()\n\n    st.dataframe(\n        attention[\n            [\n                "contract_id",\n                "supplier_name",\n                "risk_severity",\n                "risk_type",\n                "next_milestone",\n                "next_milestone_date",\n                "next_payment_amount",\n                "next_payment_currency",\n                "days_to_expiry",\n                "recommended_action",\n            ]\n        ],\n        use_container_width=True,\n        hide_index=True,\n    )\n\n    st.subheader("Risk by Supplier")\n    risk_summary = (\n        gold.groupby("supplier_name", as_index=False)\n        .agg(\n            contracts=("contract_id", "count"),\n            contract_value=("contract_value", "sum"),\n            high_risk_contracts=(\n                "risk_severity",\n                lambda x: int((x == "HIGH").sum()),\n            ),\n        )\n        .sort_values(\n            ["high_risk_contracts", "contract_value"],\n            ascending=False,\n        )\n    )\n\n    st.dataframe(\n        risk_summary,\n        use_container_width=True,\n        hide_index=True,\n    )\n\n    if DASHBOARD_URL:\n        st.link_button(\n            "Open detailed AI/BI portfolio dashboard",\n            DASHBOARD_URL,\n            use_container_width=True,\n        )\n\n\ndef render_contract_360():\n    st.header("Contract 360")\n\n    gold = load_gold()\n\n    supplier = st.selectbox(\n        "Supplier",\n        sorted(gold["supplier_name"].dropna().unique()),\n    )\n\n    supplier_contracts = gold[\n        gold["supplier_name"] == supplier\n    ].copy()\n\n    contract_id = st.selectbox(\n        "Contract",\n        supplier_contracts["contract_id"].tolist(),\n    )\n\n    record = supplier_contracts[\n        supplier_contracts["contract_id"] == contract_id\n    ].iloc[0]\n\n    c1, c2, c3, c4 = st.columns(4)\n    c1.metric("Contract Value", f"{record[\'contract_value\']:,.0f} {record[\'currency\']}")\n    c2.metric("Risk", str(record["risk_severity"]))\n    c3.metric("SLA Target", f"{record[\'sla_target_pct\']}%")\n    c4.metric("Days to Expiry", str(record["days_to_expiry"]))\n\n    c5, c6, c7 = st.columns(3)\n    c5.metric("Next Payment", f"{record[\'next_payment_amount\']:,.0f} {record[\'next_payment_currency\']}")\n    c6.metric("Penalty / Week", f"{record[\'delay_penalty_pct_per_week\']}%")\n    c7.metric("Open Obligations", str(record["open_obligations"]))\n\n    st.subheader("Operating View")\n    st.write(\n        {\n            "Next milestone": record["next_milestone"],\n            "Milestone date": record["next_milestone_date"],\n            "Payment dependency": record["next_payment_dependency"],\n            "Payment due": record["next_payment_due_date"],\n            "Risk type": record["risk_type"],\n            "Recommended action": record["recommended_action"],\n            "Renewal review due": record["renewal_review_due"],\n        }\n    )\n\n    st.subheader("Contract Evidence")\n    clauses = load_clauses()\n    selected = clauses[\n        clauses["contract_id"] == contract_id\n    ][\n        [\n            "section_number",\n            "clause_type",\n            "clause_text",\n        ]\n    ]\n\n    st.dataframe(\n        selected,\n        use_container_width=True,\n        hide_index=True,\n    )\n\n\ndef render_ask_contract_intelligence():\n    st.header("Ask Contract Intelligence")\n\n    st.caption(\n        "Answers combine governed Gold facts with Silver contract evidence."\n    )\n\n    question = st.text_area(\n        "Ask a contract question",\n        placeholder=(\n            "For CTR-2026-003, what is the main risk, "\n            "payment exposure and applicable penalty?"\n        ),\n        height=110,\n    )\n\n    if st.button(\n        "Analyze",\n        type="primary",\n        use_container_width=True,\n    ) and question.strip():\n\n        result = answer_contract_question(\n            question.strip()\n        )\n\n        st.subheader("Answer")\n        st.write(result["response"])\n\n        with st.expander("Structured Gold facts"):\n            st.dataframe(\n                result["facts"],\n                use_container_width=True,\n                hide_index=True,\n            )\n\n        with st.expander("Retrieved Silver evidence"):\n            st.dataframe(\n                result["evidence"],\n                use_container_width=True,\n                hide_index=True,\n            )\n\n        if not result["facts"].empty:\n            fact = result["facts"].iloc[0]\n\n            st.subheader("Create Follow-up Proposal")\n\n            proposed_action = st.text_area(\n                "Recommended action",\n                value=str(\n                    fact.get(\n                        "recommended_action",\n                        "Request contract owner review.",\n                    )\n                ),\n            )\n\n            owner = st.selectbox(\n                "Action owner",\n                ["Procurement", "Finance", "IT", "Operations"],\n            )\n\n            priority = st.selectbox(\n                "Priority",\n                ["P1", "P2", "P3"],\n                index=0 if fact.get("risk_severity") == "HIGH" else 1,\n            )\n\n            if st.button(\n                "Send to Human Approval",\n                use_container_width=True,\n            ):\n                evidence_text = (\n                    result["evidence"].iloc[0]["clause_text"]\n                    if not result["evidence"].empty\n                    else "No clause evidence retrieved."\n                )\n\n                approval_id = create_approval_request(\n                    contract_id=str(fact["contract_id"]),\n                    supplier_name=str(fact["supplier_name"]),\n                    proposed_action=proposed_action,\n                    owner=owner,\n                    priority=priority,\n                    rationale=str(\n                        fact.get(\n                            "risk_type",\n                            "Contract follow-up required.",\n                        )\n                    ),\n                    evidence=str(evidence_text),\n                )\n\n                st.success(\n                    f"Approval request created: {approval_id}"\n                )\n\n\ndef render_approval_inbox():\n    st.header("Approval Inbox")\n\n    approvals = query_df(\n        f"""\n        SELECT *\n        FROM {APPROVAL_TABLE}\n        ORDER BY requested_at DESC\n        """\n    )\n\n    pending = approvals[\n        approvals["approval_status"] == "PENDING"\n    ].copy()\n\n    if pending.empty:\n        st.info("No pending approvals.")\n        return\n\n    st.dataframe(\n        pending[\n            [\n                "approval_id",\n                "contract_id",\n                "supplier_name",\n                "priority",\n                "proposed_action",\n                "requested_at",\n            ]\n        ],\n        use_container_width=True,\n        hide_index=True,\n    )\n\n    approval_id = st.selectbox(\n        "Select approval request",\n        pending["approval_id"].tolist(),\n    )\n\n    record = pending[\n        pending["approval_id"] == approval_id\n    ].iloc[0]\n\n    st.write(\n        {\n            "Contract": record["contract_id"],\n            "Supplier": record["supplier_name"],\n            "Priority": record["priority"],\n            "Owner": record["action_owner"],\n            "Proposed action": record["proposed_action"],\n            "Rationale": record["rationale"],\n        }\n    )\n\n    st.subheader("Supporting Evidence")\n    st.write(record["evidence"])\n\n    approver = st.text_input(\n        "Reviewer name",\n        value="Business Reviewer",\n    )\n\n    c1, c2 = st.columns(2)\n\n    if c1.button(\n        "Approve",\n        type="primary",\n        use_container_width=True,\n    ):\n        review_approval(\n            approval_id,\n            "APPROVE",\n            approver,\n        )\n        st.cache_data.clear()\n        st.success("Approved and added to Action Queue.")\n        st.rerun()\n\n    if c2.button(\n        "Reject",\n        use_container_width=True,\n    ):\n        review_approval(\n            approval_id,\n            "REJECT",\n            approver,\n        )\n        st.cache_data.clear()\n        st.warning("Proposal rejected.")\n        st.rerun()\n\n\ndef render_action_queue():\n    st.header("Action Queue")\n\n    queue = query_df(\n        f"""\n        SELECT *\n        FROM {ACTION_QUEUE_TABLE}\n        ORDER BY queued_at DESC\n        """\n    )\n\n    if queue.empty:\n        st.info("No approved actions are currently queued.")\n        return\n\n    st.dataframe(\n        queue,\n        use_container_width=True,\n        hide_index=True,\n    )\n\n\ndef render_dashboard_page():\n    st.header("AI/BI Portfolio Dashboard")\n\n    if DASHBOARD_URL:\n        st.write(\n            "Open the detailed portfolio dashboard for "\n            "executive analytics and drill-down."\n        )\n        st.link_button(\n            "Open AI/BI Dashboard",\n            DASHBOARD_URL,\n            use_container_width=True,\n        )\n    else:\n        st.info(\n            "No dashboard URL is configured. "\n            "Set DASHBOARD_URL in app.yaml after publishing the dashboard."\n        )\n\n\nst.title("Enterprise Contract Intelligence")\nst.caption(\n    "Medallion data + evidence-backed AI + human-governed action"\n)\n\npage = st.sidebar.radio(\n    "Business Workspace",\n    [\n        "Executive Overview",\n        "Contract 360",\n        "Ask Contract Intelligence",\n        "Approval Inbox",\n        "Action Queue",\n        "AI/BI Dashboard",\n    ],\n)\n\ntry:\n    if page == "Executive Overview":\n        render_executive_overview()\n    elif page == "Contract 360":\n        render_contract_360()\n    elif page == "Ask Contract Intelligence":\n        render_ask_contract_intelligence()\n    elif page == "Approval Inbox":\n        render_approval_inbox()\n    elif page == "Action Queue":\n        render_action_queue()\n    elif page == "AI/BI Dashboard":\n        render_dashboard_page()\n\nexcept Exception as app_error:\n    st.error("The Contract Intelligence app encountered an error.")\n    st.exception(app_error)\n'

print('app.py source prepared:', len(app_py), 'characters')

In [0]:
# Define the Databricks App dependencies used by the generated Streamlit application.
requirements_txt = 'databricks-sdk\ndatabricks-sql-connector\nstreamlit\npandas\nscikit-learn\npyarrow\n'

print(requirements_txt)

In [0]:
# Define app.yaml using a SQL warehouse resource instead of hard-coding warehouse credentials or IDs.
app_yaml_template = "command: ['streamlit', 'run', 'app.py']\nenv:\n  - name: WAREHOUSE_ID\n    valueFrom: sql-warehouse\n  - name: CATALOG_NAME\n    value: '__CATALOG__'\n  - name: SCHEMA_NAME\n    value: '__SCHEMA__'\n  - name: LLM_ENDPOINT\n    value: 'system.ai.meta-llama-3-3-70b-instruct'\n  - name: DASHBOARD_URL\n    value: ''\n  - name: STREAMLIT_GATHER_USAGE_STATS\n    value: 'false'\n"

app_yaml = (
    app_yaml_template
    .replace('__CATALOG__', CATALOG)
    .replace('__SCHEMA__', SCHEMA)
)

print(app_yaml)

In [0]:
# Define a concise deployment guide that is generated alongside the Databricks App source files.
deployment_md = '# Contract Intelligence App Deployment\n\n1. Run **Lab C** and **Lab D** first.\n2. Create or select a Databricks SQL warehouse.\n3. Create a Databricks App.\n4. Add the SQL warehouse as an App resource with resource key:\n   `sql-warehouse`\n5. Grant the app service principal:\n   - `USE CATALOG` on the target catalog\n   - `USE SCHEMA` on the target schema\n   - `SELECT` on `gold_enterprise_contracts`\n   - `SELECT` on `silver_contract_clauses`\n   - `SELECT, MODIFY` on `contract_agent_approvals`\n   - `SELECT, MODIFY` on `contract_agent_action_queue`\n   - `CAN USE` on the SQL warehouse\n6. Point the app source at the generated `contract_intelligence_app` folder.\n7. Deploy the app.\n8. Share the app with business users using `CAN USE`.\n9. Publish an AI/BI dashboard and put its URL into `DASHBOARD_URL` in `app.yaml`.\n10. For enterprise rollout, surface the shared dashboard and Databricks App through Genie One.\n\nNo PAT or OAuth secret is stored in the source code. App authorization uses the Databricks App service principal.\n'

print(deployment_md)

In [0]:
# Write app.py, app.yaml, requirements.txt, and deployment instructions into a Databricks workspace folder with a local fallback.
from pathlib import Path

current_user = spark.sql(
    "SELECT current_user() AS current_user"
).first()["current_user"]

preferred_project_dir = Path(
    f"/Workspace/Users/{current_user}/contract_intelligence_app"
)

fallback_project_dir = Path(
    "/tmp/contract_intelligence_app"
)

try:
    preferred_project_dir.mkdir(
        parents=True,
        exist_ok=True
    )
    PROJECT_DIR = preferred_project_dir

except Exception:
    fallback_project_dir.mkdir(
        parents=True,
        exist_ok=True
    )
    PROJECT_DIR = fallback_project_dir

(PROJECT_DIR / "app.py").write_text(
    app_py,
    encoding="utf-8"
)

(PROJECT_DIR / "app.yaml").write_text(
    app_yaml,
    encoding="utf-8"
)

(PROJECT_DIR / "requirements.txt").write_text(
    requirements_txt,
    encoding="utf-8"
)

(PROJECT_DIR / "APP_DEPLOYMENT.md").write_text(
    deployment_md,
    encoding="utf-8"
)

print("Databricks App project created at:")
print(PROJECT_DIR)

for file_path in sorted(PROJECT_DIR.iterdir()):
    print(" -", file_path.name)


## Section 5 — Verify the Generated App

The notebook validates the generated `app.py` before deployment.

This catches Python syntax errors without requiring the Streamlit app to be started inside the notebook.


In [0]:
# Compile the generated Streamlit source and verify all expected Databricks App project files exist.
compile(
    app_py,
    "app.py",
    "exec"
)

expected_files = [
    "app.py",
    "app.yaml",
    "requirements.txt",
    "APP_DEPLOYMENT.md",
]

missing_files = [
    file_name
    for file_name in expected_files
    if not (
        PROJECT_DIR / file_name
    ).exists()
]

if missing_files:
    raise RuntimeError(
        "Missing generated app files: "
        + ", ".join(missing_files)
    )

print("app.py syntax: PASSED")
print("Project files: PASSED")


## Section 6 — Databricks App Resource Configuration

For the deployed app, add a **SQL warehouse resource** with:

```text
Resource key: sql-warehouse
Permission: CAN USE
```

The generated `app.yaml` uses:

```yaml
env:
  - name: WAREHOUSE_ID
    valueFrom: sql-warehouse
```

This keeps the application portable and avoids embedding a warehouse ID in the source code.

The app service principal also needs the minimum Unity Catalog privileges required by the POC:

```text
USE CATALOG
USE SCHEMA

SELECT:
  gold_enterprise_contracts
  silver_contract_clauses

SELECT + MODIFY:
  contract_agent_approvals
  contract_agent_action_queue
```


## Section 7 — App Authorization Model

For this POC the app uses **app authorization**.

```text
Business User
     │
     ▼
Databricks App
     │
     ▼
Dedicated App Service Principal
     │
     ├── CAN USE SQL Warehouse
     ├── SELECT Gold
     ├── SELECT Silver clauses
     └── SELECT / MODIFY approval + queue
```

No user PAT is placed in the application.

For a broader production rollout requiring per-user data entitlements, evaluate **user authorization** so the app can enforce the interacting user's own permissions rather than giving every app user the same service-principal data access.


## Section 8 — AI/BI Dashboard Integration

For the POC, keep dashboard integration simple:

1. Build and publish an AI/BI dashboard from the Gold layer.
2. Copy the published dashboard URL.
3. Add the URL to:

```yaml
- name: DASHBOARD_URL
  value: 'https://...'
```

The application then exposes:

**Open AI/BI Dashboard**

from both the Executive Overview and dedicated dashboard page.

This avoids coupling the POC to embedded-dashboard configuration while preserving a single business entry point.


## Section 9 — Genie One Enterprise Consumption

The Streamlit app is the POC business interface.

For broader enterprise consumption:

```text
                         GENIE ONE
                             │
             ┌───────────────┼─────────────────┐
             ▼               ▼                 ▼
       AI/BI Dashboard   Natural Language   Databricks App
                             Questions             │
                                                   ▼
                                       Contract Intelligence
```

Genie One becomes the simplified business-user landing experience.

Business users do not need to navigate:
- notebooks
- clusters / compute concepts
- model artifacts
- developer workspaces

They consume the shared dashboard, Genie experiences, and Databricks App from the business interface.


## Section 10 — Optional Local Project Archive

In [0]:
# Create a ZIP archive of the generated app project so the files can also be moved as one package if needed.
import shutil

archive_base = Path(
    "/tmp/contract_intelligence_app"
)

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(PROJECT_DIR)
)

print(
    "App project archive:",
    archive_path
)


## Section 11 — Business User Journey

### Executive
```text
Open App
  → Executive Overview
  → high-risk contracts
  → payment / milestone exposure
  → renewal exposure
  → detailed AI/BI Dashboard
```

### Contract Manager
```text
Open Contract 360
  → select supplier
  → select contract
  → inspect milestone / payment / SLA / penalty / evidence
```

### Procurement / Finance / Operations
```text
Ask Contract Intelligence
  → Gold structured facts
  → Silver evidence
  → AI-supported answer
  → create follow-up proposal
```

### Approver
```text
Approval Inbox
  → inspect rationale + evidence
  → APPROVE / REJECT
  → approved item enters Action Queue
```


## Section 12 — Complete POC Architecture

```text
LAB C
Sources
  → Bronze
  → AI extraction
  → Silver
  → Gold
  → Enterprise Contract View

LAB D
Gold + Silver evidence
  → Supervisor
  → Contract Agent
  → Risk Agent
  → Action Agent
  → MLflow Evals
  → Quality Gate
  → Human Approval
  → Action Queue

LAB E
Business User
  → Databricks App
      ├── Executive Overview
      ├── Contract 360
      ├── Ask Contract Intelligence
      ├── Approval Inbox
      ├── Action Queue
      └── AI/BI Dashboard
  → Genie One for broader enterprise consumption
```


In [0]:
# Print the generated business-consumption architecture and app-project location for final verification.
import json

lab_e_summary = {
    "project_directory": str(PROJECT_DIR),
    "generated_files": [
        "app.py",
        "app.yaml",
        "requirements.txt",
        "APP_DEPLOYMENT.md",
    ],
    "business_pages": [
        "Executive Overview",
        "Contract 360",
        "Ask Contract Intelligence",
        "Approval Inbox",
        "Action Queue",
        "AI/BI Dashboard",
    ],
    "data_foundation": {
        "gold": GOLD_TABLE,
        "silver_evidence": SILVER_CLAUSES,
    },
    "governance": {
        "approval_table": APPROVAL_TABLE,
        "action_queue": ACTION_QUEUE_TABLE,
    },
    "authentication": "Databricks App service principal",
    "sql_warehouse_resource_key": "sql-warehouse",
}

print(
    json.dumps(
        lab_e_summary,
        indent=2
    )
)

## Lab E Completion Checklist

### Business experience
- Executive Overview
- Contract 360
- Ask Contract Intelligence
- Approval Inbox
- Action Queue
- AI/BI Dashboard link

### Databricks App
- `app.py`
- `app.yaml`
- `requirements.txt`
- deployment instructions
- project ZIP archive

### Security
- no PAT in source
- no bearer token in source
- SQL warehouse referenced as an App resource
- app service principal authentication
- least-privilege table access documented

### Data architecture
- Gold used for structured business facts
- Silver used for contractual evidence
- approval and queue state persisted in governed Delta tables

### Enterprise consumption
- Databricks App for the POC
- AI/BI Dashboard for portfolio analytics
- Genie One as the broader business-user entry point

## Final POC sequence

**Lab C**

`Sources → Bronze → AI Extraction → Silver → Gold`

**Lab D**

`Gold + Silver → Agents → Evals → Quality Gate → Human Approval → Action Queue`

**Lab E**

`Business App → Contract 360 → Ask AI → Approval Inbox → Action Queue → Dashboard → Genie One`
